Here we will try some simple models like
1. Logistic Regression 
2. RandomForest
3. MLP

In [43]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GroupKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import confusion_matrix, accuracy_score

In [44]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [45]:
train = pd.read_csv("data/train/train_tabular.csv")

In [46]:
target_column = "failure_within_horizon"
group_column = "drone_id"

In [47]:
train = train.sort_values(['drone_id', 'flight_id']).reset_index(drop=True)

roll_features = ['payload_g', 'ambient_temp_C', 'wind_speed_ms', 'flight_duration_min', 
                 'avg_throttle', 'num_aggressive_maneuvers']

def slope(x):
    return x[-1] - x[0]

def rolling_stats(group, window):
    # group is one drone's flights as a dataframe
    # select only the columns we want to roll over
    r = group[roll_features].rolling(window, min_periods=1)
    
    mean = r.mean().fillna(0).add_suffix(f'_mean_{window}')
    std  = r.std().fillna(0).add_suffix(f'_std_{window}')
    # slp  = r.apply(slope).add_suffix(f'_slope_{window}')
    
    return pd.concat([mean, std], axis=1)

windows = [5,]

# groupby splits df into one sub-dataframe per drone
# group_keys=False stops pandas adding drone_id as an extra index level
# for each drone we compute rolling stats at each window size
# pd.concat with axis=1 joins the window results side by side
# the outer apply stitches all drones back into one dataframe
rolled = train.groupby('drone_id', group_keys=False).apply(
    lambda g: pd.concat([rolling_stats(g, w) for w in windows], axis=1)
)

# add a column counting how many flights this drone has had so far
# the model can use this to know how much history it's working with
train['flights_so_far'] = train.groupby('drone_id').cumcount() + 1

# concat joins the rolled feature columns onto df by index alignment
train = pd.concat([train.drop(columns=roll_features), rolled], axis=1)

In [48]:
train.drop( columns=["flight_id", "failure_mode"], inplace = True)

In [49]:
X = train.drop(columns = [target_column, "drone_id"])
y = train[target_column]
g = train["drone_id"]

In [50]:
categorical_columns = ["model", "motor_type", "firmware_version", "operator_region"]
numerical_columns = X.select_dtypes(include="number").columns

In [51]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
        ("num", StandardScaler(), numerical_columns),
    ]
)

In [52]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

In [53]:
models = [
    RandomForestClassifier(n_estimators=100, random_state=42),
    LogisticRegression(),
    MLPClassifier(
        hidden_layer_sizes=(5,5,2),
        random_state=42,
        max_iter = 500
    )
]

In [54]:
for model in models:
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    scores = cross_val_score(
        pipeline, X, y, groups=g, cv = GroupKFold(n_splits=5), scoring="average_precision"
    )

    print(f"Model: {model}, Mean: {scores.mean()}, Std: {scores.std()}")

Model: RandomForestClassifier(random_state=42), Mean: 0.2724128438488166, Std: 0.03349126948844732
Model: LogisticRegression(), Mean: 0.3213088873979095, Std: 0.03529146117518777
Model: MLPClassifier(hidden_layer_sizes=(5, 5, 2), max_iter=500, random_state=42), Mean: 0.12523584424360404, Std: 0.014036004459164059
